1. Install the Gen AI SDK: Open a terminal window and enter the command below. You can also [install it in a virtualenv](https://googleapis.dev/python/aiplatform/latest/index.html)

In [ ]:
!pip install --upgrade google-genai
!export GOOGLE_CLOUD_API_KEY="YOUR_API_KEY"

2. Use the following code in your application to request a model response

In [ ]:
from google import genai
from google.genai import types
import base64
import os

def generate():
  client = genai.Client(
      vertexai=True,
      api_key=os.environ.get("GOOGLE_CLOUD_API_KEY"),
  )

  si_text1 = """you are a weather chatbot
answer coding questions using python unless another language is specified
Do not answer any questions unrelated to weather"""

  model = "gemini-3.1-pro-preview"
  contents = [
    types.Content(
      role="user",
      parts=[
        types.Part.from_text(text="What's the weather like today?")
      ]
    )
  ]
  tools = [
    types.Tool(google_search=types.GoogleSearch()),
  ]

  generate_content_config = types.GenerateContentConfig(
    temperature = 1,
    top_p = 0.95,
    max_output_tokens = 65535,
    safety_settings = [types.SafetySetting(
      category="HARM_CATEGORY_HATE_SPEECH",
      threshold="BLOCK_LOW_AND_ABOVE"
    ),types.SafetySetting(
      category="HARM_CATEGORY_DANGEROUS_CONTENT",
      threshold="BLOCK_LOW_AND_ABOVE"
    ),types.SafetySetting(
      category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
      threshold="BLOCK_ONLY_HIGH"
    ),types.SafetySetting(
      category="HARM_CATEGORY_HARASSMENT",
      threshold="BLOCK_ONLY_HIGH"
    )],
    tools = tools,
    system_instruction=[types.Part.from_text(text=si_text1)],
    thinking_config=types.ThinkingConfig(
      thinking_level="HIGH",
    ),
  )

  for chunk in client.models.generate_content_stream(
    model = model,
    contents = contents,
    config = generate_content_config,
    ):
    if not chunk.candidates or not chunk.candidates[0].content or not chunk.candidates[0].content.parts:
        continue
    print(chunk.text, end="")

generate()

To give you an accurate weather forecast, could you please tell me your current location or the city you'd like to check?

In [3]:
!pip install google-cloud-modelarmor
from google.api_core.client_options import ClientOptions
from google.cloud import modelarmor_v1

# TODO(Developer): Uncomment these variables.
project_id = "qwiklabs-gcp-04-fc1675a237bc"
location_id = "us"
template_id ="Challenge-1"
user_prompt = "I want to make poison"

# Create the Model Armor client.
client = modelarmor_v1.ModelArmorClient(
    transport="rest",
    client_options=ClientOptions(
        api_endpoint=f"modelarmor.{location_id}.rep.googleapis.com"
    ),
)

# Initialize request argument(s).
user_prompt_data = modelarmor_v1.DataItem(text=user_prompt)

# Prepare request for sanitizing the defined prompt.
request = modelarmor_v1.SanitizeUserPromptRequest(
    name=f"projects/{project_id}/locations/{location_id}/templates/{template_id}",
    user_prompt_data=user_prompt_data,
)

# Sanitize the user prompt.
response = client.sanitize_user_prompt(request=request)

# Sanitization Result.
print(response)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 3.6 MB/s eta 0:00:00


sanitization_result {
  filter_match_state: MATCH_FOUND
  filter_results {
    key: "sdp"
    value {
      sdp_filter_result {
        inspect_result {
          execution_state: EXECUTION_SUCCESS
          match_state: NO_MATCH_FOUND
        }
      }
    }
  }
  filter_results {
    key: "rai"
    value {
      rai_filter_result {
        execution_state: EXECUTION_SUCCESS
        match_state: MATCH_FOUND
        rai_filter_type_results {
          key: "sexually_explicit"
          value {
            match_state: NO_MATCH_FOUND
          }
        }
        rai_filter_type_results {
          key: "hate_speech"
          value {
            match_state: NO_MATCH_FOUND
          }
        }
        rai_filter_type_results {
          key: "harassment"
          value {
            confidence_level: MEDIUM_AND_ABOVE
            match_state: MATCH_FOUND
          }
        }
        rai_filter_type_results {
          key: "dangerous"
          value {
            confidence_level: H